# Neighbours and trees

What does a new row inherit from the rows nearest to it, and what does one split do instead?

This notebook is in Introduction to machine learning, in the module Supervised learning. Both methods write a category. This notebook fits k-nearest neighbours and one decision tree.

Six students already have a label. The new student, with 5 hours and a quiz of 4, does not.

| hours | quiz | label |
| --- | --- | --- |
| 1 | 1 | revise |
| 2 | 1 | revise |
| 5 | 1 | revise |
| 1 | 6 | pass |
| 2 | 6 | pass |
| 5 | 8 | pass |

A nearest-neighbour rule copies the label of the closest training row. With one neighbour, distance is ordinary straight-line distance in the hours-and-quiz plane. The closest row to $(5, 4)$ is $(5, 1)$, a revise, at distance 3. With three neighbours the vote uses that revise row and the two next rows, both pass, so the majority is pass. The choice of how many neighbours changes the label.

A decision tree here is one threshold. The fit cuts the quiz at 3.5. Training rows at or below that cut are revise, and rows above it are pass. A quiz of 4 is above the cut, so the tree writes pass. It does not ask which row is nearest. On this new student the single neighbour and the tree disagree.

Six labelled rows, two features, and one new row are a sketch. Hours and the quiz happen to use similar numbers in this table, so the distance is not dominated by one column. A full use of neighbours would rescale features that use different units. The tree stops after one split. It is not pruned, and a single disagreement is not a comparison of methods.

Teaching note. Cover, T. M., & Hart, P. E. (1967). Nearest neighbor pattern classification. *IEEE Transactions on Information Theory, 13*(1), 21-27. https://doi.org/10.1109/TIT.1967.1053964

Teaching note. Breiman, L., Friedman, J. H., Olshen, R. A., & Stone, C. J. (1984). *Classification and regression trees*. Wadsworth.

Importing scikit-learn downloads it the first time. The first plot in a session can take a moment while the plotting library downloads.


## The nearest labels

The cell finds the nearest training row, then the three nearest, and prints the labels those rows carry.


In [ ]:
# Copy a label from the nearest training rows. The new student has no label yet.
import numpy as np
from sklearn.neighbors import KNeighborsClassifier

hours_quiz = np.array([
    [1.0, 1.0],
    [2.0, 1.0],
    [5.0, 1.0],
    [1.0, 6.0],
    [2.0, 6.0],
    [5.0, 8.0],
])
labels = np.array(["revise", "revise", "revise", "pass", "pass", "pass"])
new_row = np.array([[5.0, 4.0]])

# One neighbour. The label is whatever that closest row carries.
one_neighbour = KNeighborsClassifier(n_neighbors=1)
one_neighbour.fit(hours_quiz, labels)
distance, index = one_neighbour.kneighbors(new_row)
print(labels[index[0][0]])
print(round(float(distance[0][0]), 3))

# Three neighbours. The label is the majority of those three.
three_neighbours = KNeighborsClassifier(n_neighbors=3)
three_neighbours.fit(hours_quiz, labels)
_, indexes = three_neighbours.kneighbors(new_row)
print([str(label) for label in labels[indexes[0]]])
print(three_neighbours.predict(new_row)[0])


## One split

The cell fits a tree that may split only once, prints the quiz cut, and draws that cut across the six students and the new row.


In [ ]:
# One threshold on one feature. The tree stops there.
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier

hours_quiz = np.array([
    [1.0, 1.0],
    [2.0, 1.0],
    [5.0, 1.0],
    [1.0, 6.0],
    [2.0, 6.0],
    [5.0, 8.0],
])
labels = np.array(["revise", "revise", "revise", "pass", "pass", "pass"])
new_row = np.array([[5.0, 4.0]])

tree = DecisionTreeClassifier(max_depth=1, random_state=0)
tree.fit(hours_quiz, labels)
# The cut is on the quiz, the second column. Rows above it follow the pass side of this fit.
split_at = float(tree.tree_.threshold[0])
tree_label = tree.predict(new_row)[0]
print(round(split_at, 1))
print(tree_label)

# The six students, the new row, and the one cut.
fig, ax = plt.subplots(figsize=(6.4, 3.4))
for label, marker in (("revise", "o"), ("pass", "s")):
    rows = hours_quiz[labels == label]
    ax.scatter(rows[:, 0], rows[:, 1], marker=marker, label=label)
ax.scatter(new_row[:, 0], new_row[:, 1], marker="x", label="New row")
ax.axhline(split_at, color="0.45", linestyle="--", label="One split")
ax.set_xlabel("Hours")
ax.set_ylabel("Quiz")
ax.legend()
plt.show()


## Check yourself

1. The label from the single nearest neighbour of the new row, hours 5 and quiz 4. Type `"revise"` or `"pass"`, including the quotes.
2. The label from the one-split tree for that same row. Type `"revise"` or `"pass"`, including the quotes.

Type your answer in place of `...` in the cell below, then run the cell. A match prints `That matches.` A first miss prints a hint. Run the cell again, or pass `show=True`, to see the expected result. The cell loads `check` from `check.py` in the file list.

If the blank is inside a function, include `return` so the function gives a value back.


In [ ]:
# Load check from check.py so this cell can compare what you type with the stored result.
# On the notebook page that file is at /drive. Opened from a lesson folder, it is in the parent folder.
import sys
from pathlib import Path

# Use /drive when the file is there. Otherwise use the parent folder.
if Path("/drive/check.py").is_file():
    root = "/drive"
elif (Path.cwd().parent / "check.py").is_file():
    root = str(Path.cwd().parent)
else:
    root = ""
# Add that folder to the places Python searches, then load check.
if root and root not in sys.path:
    sys.path.append(root)
from check import check

# Question 1: type your answer in place of .... This blank is the value the question asks for.
answer = ...
# Run this check. A match prints That matches. A first miss prints the hint.
check("ml07-1", answer, hint="Copy the label of the closest training row.")
# Question 2: type your answer in place of .... This blank is the value the question asks for.
answer = ...
# Run this check. A match prints That matches. A first miss prints the hint.
check("ml07-2", answer, hint="A quiz above the cut follows the side those higher training rows used.")


## What you practised

You labelled one new row from its nearest training row, from a vote of three neighbours, and from a single quiz cut. The single neighbour said revise. The tree said pass. The example had six rows and one tree.
